## Building Unit Cleaning Code, for San Diego County

Used for joining with SDGE utility

In [1]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import fiona
import pandas as pd

# statistical libraries
#import sys
#!{sys.executable} -m pip install statsmodels
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

In [2]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

In [3]:
# load parcels 
parcels = gpd.read_parquet("/../../capstone/electrigrid/data/parcels/parcels_final.parquet").to_crs(epsg=4326)

# filter for San Diego
parcels = parcels[parcels['County'] == "San Diego"]

In [4]:
# import Zillow data 
zillow = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

Building Footprint

Access: https://sat-io.earthengine.app/view/gba

In [5]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
building = gpd.read_parquet("../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet").to_crs(epsg=4326)

In [6]:
# load in SDG&E boundary map
sdge_shape = gpd.read_file("/../../capstone/electrigrid/data/utilities/IOU_shapefiles.geojson")

sdge_shape = sdge_shape[sdge_shape['Acronym'] == "SDG&E"]

## Unit-regression for multi-family homes

### Select parcels for multi-family homes

In [125]:
# only keep the geometry and parcel number of parcels
parcels = parcels[['PARNO', 'geometry']]

## select only multi-family data from Zillow
zillow_multi = zillow[zillow['type'] == "Multi"]
zillow_multi = zillow_multi[zillow_multi['code'] != "RR106"]

## crop only to residential parcels
# keep the indices where multi-family homes match to parcels
valid_parcels = parcels.sjoin(zillow_multi, predicate="contains").index.unique()

# select the parcels that match these indices
parcels_res = parcels.loc[valid_parcels]

# confirm that joining with Zillow decreases the number of parcels
assert len(parcels_res) < len(parcels)

### Investigative analysis
Some of the parcels have more than one Zillow point within them so let's add code to find where multiple points are within one polygon to add the units together before dropping the duplicate units. 

In [126]:
# Count points in polygons
zillow_in_parcels = (
        # Spatial join associates points and polygons that intersects each other
        parcels.sjoin(
            zillow_multi,
            how="inner",  # Only keep points that matches a polygon
        )
        .groupby('PARNO')  # Group points by polygons
        .size()  # Get number of points
        .rename('num_zillow_points')  # Name your column as you want it to appear in polygons
    )

# investigate the parcels where more than one zillow point exists
multiple_zillows = zillow_in_parcels[zillow_in_parcels > 1]
print(f"There are ", len(multiple_zillows)," parcels with more than one Zillow point within a parcel.")

There are  6945  parcels with more than one Zillow point within a parcel.


### Select buildings that are multi-family homes and add Zillow data
Adding the units_sum to the parcel_res accounts for the multiple Zillow point that can occur in one parcel. These values will be utilized going forward as we will be computing the regression for the total_volume of the buildings in one parcel against the total number of units within that parcel. 

In [127]:

# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(parcels_res, predicate="intersects").index.unique()
buildings_res = building.loc[valid_buildings]

## join parcels to buildings (keeping observations as parcels, but with building attributes)
# sum number of units per parcel
units_sum = parcels_res.sjoin(zillow_multi, predicate="intersects").groupby(level=0)["unit"].sum()

# join on parcels with summed number of units
parcels_res = parcels_res.join(units_sum)

# confirm that joining with Zillow decreased the number of buildings
assert len(buildings_res) < len(building)

# keep all residential buildings, and add zillow points only where they match up
building_zillow = gpd.sjoin(
    buildings_res,
    zillow_multi,
    predicate = "intersects")

building_zillow = building_zillow.drop(columns = "index_right")

In [16]:
parcels_res.head()

,PARNO,geometry,unit
7629947,4241720700,"MULTIPOLYGON Z (((-117.23498 32.79751 0.00000,...",3.0
7630042,4241721300,"MULTIPOLYGON Z (((-117.23439 32.79723 0.00000,...",2.0
7630043,4241722000,"MULTIPOLYGON Z (((-117.23549 32.79699 0.00000,...",3.0
7630075,4236921300,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",2.0
7630078,4236220200,"MULTIPOLYGON Z (((-117.25236 32.77776 0.00000,...",3.0


### Calculate the volume and aggregate based on parcels
The problem I think that's happening here is that the units are added to every single building. When multiple buildings are in one parcel the units are double counted. We need to add in the parcel number and either aggregate to the parcel.

In [128]:
# add in the parcel number to add all of the building volumes for the regression
building_zillow_parcels = building_zillow.sjoin(
                                parcels_res,
                                how = "left",
                                predicate = "intersects")

# drop the unnecessary columns
building_zillow_parcels = building_zillow_parcels.drop(columns = ['index_right', 'unit_left'])
building_zillow_parcels = building_zillow_parcels.rename(columns = {'unit_right': 'total_unit'})

In [18]:
building_zillow_parcels.head()

,source,id,height,var,region,bbox,geometry,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,total_unit
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-116.99693 33.29913, -116.99687 33.2...",Multi,2009.0,4.0,None,None,None,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,1323607700,2.0
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-116.98655 33.25268, -116.98620 33.2...",Multi,1985.0,5.0,None,None,O,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,1881920200,2.0
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-116.37161 33.25203, -116.37152 33.2...",Multi,1998.0,4.0,None,None,None,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612100,2.0
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-116.37224 33.25284, -116.37224 33.2...",Multi,1959.0,4.0,None,None,None,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,1980522300,2.0
6353689,osm,1136229915,3.878115,0.254450,USA,"{'xmax': -116.37282159999998, 'xmin': -116.372...","POLYGON ((-116.37297 33.25286, -116.37283 33.2...",Multi,1962.0,4.0,None,None,None,155077.0,living,1736.0,7599626,06073021002,h479,SDGE,RI101,1980522600,2.0


In [129]:
# reproject data frame to crs with meters as units
building_m = building_zillow_parcels.to_crs("EPSG:6933")

# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

In [20]:
building_m.head()

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,total_unit,area_m2,volume_m3
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-11288598.440 4018349.084, -11288592...",Multi,2009.0,4.0,None,None,None,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,1323607700,2.0,412.421351,1606.752247
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-11287597.377 4013374.612, -11287563...",Multi,1985.0,5.0,None,None,O,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,1881920200,2.0,452.260448,1858.907171
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612100,2.0,232.343850,905.530915
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-11228325.006 4013391.447, -11228324...",Multi,1959.0,4.0,None,None,None,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,1980522300,2.0,284.844083,1177.213436
6353689,osm,1136229915,3.878115,0.254450,USA,"{'xmax': -116.37282159999998, 'xmin': -116.372...","POLYGON ((-11228395.315 4013393.814, -11228381...",Multi,1962.0,4.0,None,None,None,155077.0,living,1736.0,7599626,06073021002,h479,SDGE,RI101,1980522600,2.0,342.459460,1328.097228


In [130]:
# keep only observations with unit data
building_w_units = building_m[~building_m['total_unit'].isna()]

assert building_w_units['total_unit'].isna().sum() == 0

There are also some multi-family homes where the units equal zero. Let's drop those so that there we can also predict them based on the regression.

In [131]:
# drop multi-family homes where the total_unit is equal to zero
building_w_units = building_w_units.drop(building_w_units[building_w_units['total_unit'] == 0].index)

Aggregate the volumes for the unit regression by parcel.

In [132]:
# aggregate the volumes by parcel
total_volume_m3 = building_w_units.groupby('PARNO')['volume_m3'].sum(min_count = 1)

In [133]:
# change the series to a dataframe
total_volume_m3 = pd.DataFrame(total_volume_m3)

# rename the column to total_volume_m3
total_volume_m3 = total_volume_m3.rename(columns = {'volume_m3' : 'total_volume_m3'})

# add the total volume to the buildings dataframe
building_w_units = building_w_units.join(total_volume_m3, on = 'PARNO')

In [25]:
building_w_units.head()

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,total_unit,area_m2,volume_m3,total_volume_m3
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-11288598.440 4018349.084, -11288592...",Multi,2009.0,4.0,None,None,None,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,1323607700,2.0,412.421351,1606.752247,1606.752247
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-11287597.377 4013374.612, -11287563...",Multi,1985.0,5.0,None,None,O,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,1881920200,2.0,452.260448,1858.907171,1858.907171
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612100,2.0,232.343850,905.530915,905.530915
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-11228325.006 4013391.447, -11228324...",Multi,1959.0,4.0,None,None,None,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,1980522300,2.0,284.844083,1177.213436,1177.213436
6353689,osm,1136229915,3.878115,0.254450,USA,"{'xmax': -116.37282159999998, 'xmin': -116.372...","POLYGON ((-11228395.315 4013393.814, -11228381...",Multi,1962.0,4.0,None,None,None,155077.0,living,1736.0,7599626,06073021002,h479,SDGE,RI101,1980522600,2.0,342.459460,1328.097228,1328.097228


The parcel data is from 2014 while the Zillow data is from 2025. There are some Zillow points not in parcels which is assumed to be due to the differences in time frames from the data. Parcels were likely added after 2014. 

In [134]:
# some of the homes don't have a parcel number (so replace the total volume with volume if its na
building_w_units['total_volume_m3'] = building_w_units['total_volume_m3'].fillna(building_w_units['volume_m3'])

In [27]:
building_w_units.head()

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,total_unit,area_m2,volume_m3,total_volume_m3
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-11288598.440 4018349.084, -11288592...",Multi,2009.0,4.0,None,None,None,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,1323607700,2.0,412.421351,1606.752247,1606.752247
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-11287597.377 4013374.612, -11287563...",Multi,1985.0,5.0,None,None,O,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,1881920200,2.0,452.260448,1858.907171,1858.907171
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612100,2.0,232.343850,905.530915,905.530915
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-11228325.006 4013391.447, -11228324...",Multi,1959.0,4.0,None,None,None,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,1980522300,2.0,284.844083,1177.213436,1177.213436
6353689,osm,1136229915,3.878115,0.254450,USA,"{'xmax': -116.37282159999998, 'xmin': -116.372...","POLYGON ((-11228395.315 4013393.814, -11228381...",Multi,1962.0,4.0,None,None,None,155077.0,living,1736.0,7599626,06073021002,h479,SDGE,RI101,1980522600,2.0,342.459460,1328.097228,1328.097228


In [135]:
# remove duplicates from the parcel number so that it doesn't skew the linear regression
building_w_units = building_w_units.drop_duplicates(subset = 'PARNO', keep = 'first')

### Run the linear regression, remove outliers, and re-run the regression

In [136]:
# run model
results = smf.ols('total_unit ~ total_volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

In [30]:
building_w_units.head()

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,total_unit,area_m2,volume_m3,total_volume_m3,residual
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-11288598.440 4018349.084, -11288592...",Multi,2009.0,4.0,None,None,None,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,1323607700,2.0,412.421351,1606.752247,1606.752247,-44.187476
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-11287597.377 4013374.612, -11287563...",Multi,1985.0,5.0,None,None,O,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,1881920200,2.0,452.260448,1858.907171,1858.907171,-44.272130
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612100,2.0,232.343850,905.530915,905.530915,-43.952060
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-11228325.006 4013391.447, -11228324...",Multi,1959.0,4.0,None,None,None,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,1980522300,2.0,284.844083,1177.213436,1177.213436,-44.043270
6353689,osm,1136229915,3.878115,0.254450,USA,"{'xmax': -116.37282159999998, 'xmin': -116.372...","POLYGON ((-11228395.315 4013393.814, -11228381...",Multi,1962.0,4.0,None,None,None,155077.0,living,1736.0,7599626,06073021002,h479,SDGE,RI101,1980522600,2.0,342.459460,1328.097228,1328.097228,-44.093925


In [137]:
# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

In [138]:
print(f"There are ", building_units_clean.shape[0], "buildings included in the regression.")
print(f"There are ", building_outliers.shape[0], "outliers")

There are  31158 buildings included in the regression.
There are  2584 outliers


In [139]:
# rerun linear regression
results_clean = smf.ols('total_unit ~ total_volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]

/tmp/ipykernel_2518103/860630456.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept = results_clean.params[0]
/tmp/ipykernel_2518103/860630456.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope = results_clean.params[1]


In [64]:
intercept

24.062091744455024

In [65]:
slope

0.00036052712157647293

### Use the values from the linear regression to compute missing units

In [140]:
# extract just the multi-family homes where unit info is missing
missing_units = building_m[(building_m['total_unit'].isna()) & (building_m['total_unit'] == 0)]


# aggregate the volumes for the unit regression by parcel
missing_total_volume_m3 = missing_units.groupby('PARNO')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
missing_total_volume_m3 = pd.DataFrame(missing_total_volume_m3)

# rename the column to missing_total_volume_m3
missing_total_volume_m3 = missing_total_volume_m3.rename(columns = {'volume_m3' : 'missing_total_volume_m3'})

# add the missing total volume to the buildings dataframe
missing_units = missing_units.join(missing_total_volume_m3, on = 'PARNO')

In [141]:
# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units])

assert len(missing_units) < len(missing_outlier_units)

In [142]:
# replace anywhere the missing_total_volume is missing (fill in for the outliers since total_volume was computed before)
missing_outlier_units['missing_total_volume_m3'] = missing_outlier_units['missing_total_volume_m3'].fillna(missing_outlier_units['total_volume_m3'])

In [39]:
missing_outlier_units.head()

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,total_unit,area_m2,volume_m3,total_volume_m3,residual,missing_total_volume_m3
6391517,ms,UnitedStates_023013221_176182,5.072415,0.830749,USA,"{'xmax': -117.07446459411786, 'xmin': -117.074...","POLYGON ((-11296079.601 3986444.034, -11296088...",Multi,1988.0,NaN,None,None,None,125090245.0,None,NaN,7842680,06073017052,h575,SDGE,RI104,3134210300,330.0,265.145315,1344.927166,1344.927166,283.900425,1344.927166
6402094,ms,UnitedStates_023013221_427657,7.921669,1.541906,USA,"{'xmax': -117.0882616934217, 'xmin': -117.0885...","POLYGON ((-11297441.441 3989805.056, -11297419...",Multi,1990.0,NaN,None,None,None,116253770.0,None,NaN,8357792,06073017063,h519,SDGE,RI104,6783300500,402.0,415.112063,3288.380366,3288.380366,355.247964,3288.380366
6488472,osm,1308345952,9.071153,2.293343,USA,"{'xmax': -117.27988379999996, 'xmin': -117.280...","POLYGON ((-11315950.285 3998191.696, -11315947...",Multi,NaN,NaN,None,None,None,66203762.0,None,NaN,7642279,06073022102,h589,SDGE,RI104,2158400300,288.0,1020.570063,9257.746873,9257.746873,239.243915,9257.746873
6500620,osm,1279771192,7.621391,1.815493,USA,"{'xmax': -117.2999662, 'xmin': -117.3002135999...","POLYGON ((-11317851.683 4005724.897, -11317849...",Multi,2018.0,NaN,None,None,None,10189716.0,living,67255.0,8415568,06073019803,h491,SDGE,RI104,1670402100,342.0,367.381136,2799.955392,24788.418429,288.029923,24788.418429
6507893,osm,1273358220,9.522633,2.515816,USA,"{'xmax': -117.32649259999997, 'xmin': -117.326...","POLYGON ((-11320432.883 4004939.992, -11320425...",Multi,1985.0,NaN,None,None,None,83342588.0,None,NaN,7552203,06073017801,h491,SDGE,RI104,1672504400,300.0,384.902388,3665.284026,3665.284026,253.121429,3665.284026


In [143]:
# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('total_unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

missing_outlier_units_pred['total_unit'] = round(intercept + missing_outlier_units_pred['missing_total_volume_m3'] * slope)

In [41]:
# range of calculated units
missing_outlier_units_pred['total_unit'].agg(['min','max'])

min     24.0
max    164.0
Name: total_unit, dtype: float64

In [144]:
missing_outlier_units_pred = missing_outlier_units_pred.drop_duplicates(subset = 'PARNO', keep = 'first')

In [146]:
# combine multi-family homes data frames
multi_complete = pd.concat([building_w_units, missing_outlier_units_pred]).to_crs(zillow.crs)

# drop residual column
multi_complete = multi_complete.drop(['residual', 'geometry'], axis = 1)

# drop the unit data from parcel res as the new unit data was computed
parcels_res = parcels_res.drop(columns = ['unit'])

In [147]:
# join the parcel data on PARNO to get the parcel geometry
multi_by_parcel = parcels_res.merge(multi_complete, on = 'PARNO')

In [79]:
parcels_res.head()

,PARNO,geometry
7629947,4241720700,"MULTIPOLYGON Z (((-117.23498 32.79751 0.00000,..."
7630042,4241721300,"MULTIPOLYGON Z (((-117.23439 32.79723 0.00000,..."
7630043,4241722000,"MULTIPOLYGON Z (((-117.23549 32.79699 0.00000,..."
7630075,4236921300,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,..."
7630078,4236220200,"MULTIPOLYGON Z (((-117.25236 32.77776 0.00000,..."


In [80]:
multi_complete.head()

,PARNO,geometry,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,total_unit,area_m2,volume_m3,total_volume_m3,missing_total_volume_m3
0,4236921300,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,2.0,140.235708,431.200368,431.200368,NaN
1,4236220200,"MULTIPOLYGON Z (((-117.25236 32.77776 0.00000,...",ms,UnitedStates_023013221_128769,2.718830,0.218748,USA,"{'xmax': -117.2521855055026, 'xmin': -117.2522...",Multi,1980.0,5.0,None,None,None,986647.0,living,1932.0,7995754,06073007601,h530,SDGE,RI101,3.0,117.733224,320.096607,320.096607,NaN
2,4245420100,"MULTIPOLYGON Z (((-117.23302 32.79141 0.00000,...",ms,UnitedStates_023013221_369659,7.822832,2.910877,USA,"{'xmax': -117.23303161924132, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,158049.0,None,NaN,7998882,06073007702,h530,SDGE,RI110,4.0,224.418227,1755.586108,3445.225886,NaN
3,4245420700,"MULTIPOLYGON Z (((-117.23280 32.79066 0.00000,...",ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,2.0,113.884017,734.439912,734.439912,NaN
4,5670903600,"MULTIPOLYGON Z (((-117.09493 32.63592 0.00000,...",osm,1182648717,6.608080,1.040565,USA,"{'xmax': -117.09509669999997, 'xmin': -117.095...",Multi,1968.0,NaN,None,None,None,5433899.0,living,25844.0,8226026,06073012501,h564,SDGE,RI104,40.0,1895.486015,12525.523059,12525.523059,NaN


### Remove duplicates and make centroids for multi-family homes
Multi-family homes are in polygons. In order to combine the multi-family and single family data we need to make them points. The unit linear regression calculations were completed for buildings so there are multiples of each polygon. We only need one row per parcel to make the centroids. 

In [148]:
# check how many parcel numbers are duplicates
duplicates = multi_by_parcel.pivot_table(index = ['PARNO'], aggfunc = 'size')
print(f"There are ",duplicates[duplicates > 1].sum(), " duplicated parcels.")

There are  5168  duplicated parcels.


**Remove duplicates so that there aren't duplicate points. The units are aggregated by parcel because they were run on the total_volume.**

In [149]:
# remove duplicates from the parcel number so we don't get multiple points per parcel
multi_by_parcel = multi_by_parcel.drop_duplicates(subset = 'PARNO', keep = 'first')

In [112]:
multi_by_parcel.head()

,PARNO,geometry,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,total_unit,area_m2,volume_m3,total_volume_m3,missing_total_volume_m3
0,4236921300,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,2.0,140.235708,431.200368,431.200368,NaN
1,4236220200,"MULTIPOLYGON Z (((-117.25236 32.77776 0.00000,...",ms,UnitedStates_023013221_128769,2.718830,0.218748,USA,"{'xmax': -117.2521855055026, 'xmin': -117.2522...",Multi,1980.0,5.0,None,None,None,986647.0,living,1932.0,7995754,06073007601,h530,SDGE,RI101,3.0,117.733224,320.096607,320.096607,NaN
2,4245420100,"MULTIPOLYGON Z (((-117.23302 32.79141 0.00000,...",ms,UnitedStates_023013221_369659,7.822832,2.910877,USA,"{'xmax': -117.23303161924132, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,158049.0,None,NaN,7998882,06073007702,h530,SDGE,RI110,4.0,224.418227,1755.586108,3445.225886,NaN
3,4245420700,"MULTIPOLYGON Z (((-117.23280 32.79066 0.00000,...",ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,2.0,113.884017,734.439912,734.439912,NaN
4,5670903600,"MULTIPOLYGON Z (((-117.09493 32.63592 0.00000,...",osm,1182648717,6.608080,1.040565,USA,"{'xmax': -117.09509669999997, 'xmin': -117.095...",Multi,1968.0,NaN,None,None,None,5433899.0,living,25844.0,8226026,06073012501,h564,SDGE,RI104,40.0,1895.486015,12525.523059,12525.523059,NaN


In [150]:
# set the geometry to the parcel geometry
multi_by_parcel = multi_by_parcel.set_geometry('geometry')

# change the crs 
multi_by_parcel = multi_by_parcel.to_crs("EPSG:6933")

In [151]:
## IN ORDER TO CONCAT THE MULTI-FAMILY HOMES WITH SINGLE FAMILY HOMES THEY MUST ALL BE POINTS
# create centroids for each multi residential parcel 
multi_by_parcel['centroids'] = multi_by_parcel.geometry.centroid


In [116]:
multi_by_parcel.head()

,PARNO,geometry,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,total_unit,area_m2,volume_m3,total_volume_m3,missing_total_volume_m3,centroids
0,4236921300,MULTIPOLYGON Z (((-11313159.182 3961251.281 0....,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,2.0,140.235708,431.200368,431.200368,NaN,POINT (-11313153.723 3961239.229)
1,4236220200,MULTIPOLYGON Z (((-11313244.013 3962367.862 0....,ms,UnitedStates_023013221_128769,2.718830,0.218748,USA,"{'xmax': -117.2521855055026, 'xmin': -117.2522...",Multi,1980.0,5.0,None,None,None,986647.0,living,1932.0,7995754,06073007601,h530,SDGE,RI101,3.0,117.733224,320.096607,320.096607,NaN,POINT (-11313238.717 3962356.234)
2,4245420100,MULTIPOLYGON Z (((-11311377.586 3963837.438 0....,ms,UnitedStates_023013221_369659,7.822832,2.910877,USA,"{'xmax': -117.23303161924132, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,158049.0,None,NaN,7998882,06073007702,h530,SDGE,RI110,4.0,224.418227,1755.586108,3445.225886,NaN,POINT (-11311395.134 3963841.243)
3,4245420700,MULTIPOLYGON Z (((-11311356.480 3963757.406 0....,ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,2.0,113.884017,734.439912,734.439912,NaN,POINT (-11311373.489 3963745.624)
4,5670903600,MULTIPOLYGON Z (((-11298054.295 3947082.794 0....,osm,1182648717,6.608080,1.040565,USA,"{'xmax': -117.09509669999997, 'xmin': -117.095...",Multi,1968.0,NaN,None,None,None,5433899.0,living,25844.0,8226026,06073012501,h564,SDGE,RI104,40.0,1895.486015,12525.523059,12525.523059,NaN,POINT (-11298091.345 3947127.302)


In [152]:

# drop the geometry of multi_by_parcel so the centroid becomes the geometry
multi_by_parcel = multi_by_parcel.drop(columns = 'geometry')

# rename the centroid to geometry
multi_by_parcel = multi_by_parcel.rename(columns = {'centroids' : 'geometry'})

# set the centroid to the geometry
multi_by_parcel_processed = multi_by_parcel.set_geometry('geometry')

In [119]:
multi_by_parcel_processed.head()

,PARNO,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,total_unit,area_m2,volume_m3,total_volume_m3,missing_total_volume_m3,geometry
0,4236921300,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,2.0,140.235708,431.200368,431.200368,NaN,POINT (-11313153.723 3961239.229)
1,4236220200,ms,UnitedStates_023013221_128769,2.718830,0.218748,USA,"{'xmax': -117.2521855055026, 'xmin': -117.2522...",Multi,1980.0,5.0,None,None,None,986647.0,living,1932.0,7995754,06073007601,h530,SDGE,RI101,3.0,117.733224,320.096607,320.096607,NaN,POINT (-11313238.717 3962356.234)
2,4245420100,ms,UnitedStates_023013221_369659,7.822832,2.910877,USA,"{'xmax': -117.23303161924132, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,158049.0,None,NaN,7998882,06073007702,h530,SDGE,RI110,4.0,224.418227,1755.586108,3445.225886,NaN,POINT (-11311395.134 3963841.243)
3,4245420700,ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,2.0,113.884017,734.439912,734.439912,NaN,POINT (-11311373.489 3963745.624)
4,5670903600,osm,1182648717,6.608080,1.040565,USA,"{'xmax': -117.09509669999997, 'xmin': -117.095...",Multi,1968.0,NaN,None,None,None,5433899.0,living,25844.0,8226026,06073012501,h564,SDGE,RI104,40.0,1895.486015,12525.523059,12525.523059,NaN,POINT (-11298091.345 3947127.302)


### Adjust single-family Zillow data to the SDG&E area and adjust values

In [153]:
# change the CRS back to the Zillow CRS 
multi_by_parcel_processed = multi_by_parcel_processed.to_crs(zillow.crs)

# subset for single family zillow and condos
single_zillow = zillow[zillow['type'] != "Single"].to_crs(zillow.crs)

# keep only single family homes within San Diego County
single_zillow = single_zillow.clip(sdge_shape)

# change all single_zillow to unit = 1
single_zillow['total_unit'] = 1

# drop the unit column
single_zillow = single_zillow.drop(['unit'], axis = 1)

# concat the single and multifamily dataframes
complete_zillow = pd.concat([multi_by_parcel_processed, single_zillow], axis = 0)

In [121]:
complete_zillow.head()

,PARNO,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,total_unit,area_m2,volume_m3,total_volume_m3,missing_total_volume_m3,geometry,unit
0,4236921300,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,2.0,140.235708,431.200368,431.200368,NaN,POINT (-117.25142 32.76728),NaN
1,4236220200,ms,UnitedStates_023013221_128769,2.718830,0.218748,USA,"{'xmax': -117.2521855055026, 'xmin': -117.2522...",Multi,1980.0,5.0,None,None,None,986647.0,living,1932.0,7995754,06073007601,h530,SDGE,RI101,3.0,117.733224,320.096607,320.096607,NaN,POINT (-117.25230 32.77765),NaN
2,4245420100,ms,UnitedStates_023013221_369659,7.822832,2.910877,USA,"{'xmax': -117.23303161924132, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,158049.0,None,NaN,7998882,06073007702,h530,SDGE,RI110,4.0,224.418227,1755.586108,3445.225886,NaN,POINT (-117.23320 32.79144),NaN
3,4245420700,ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,2.0,113.884017,734.439912,734.439912,NaN,POINT (-117.23297 32.79055),NaN
4,5670903600,osm,1182648717,6.608080,1.040565,USA,"{'xmax': -117.09509669999997, 'xmin': -117.095...",Multi,1968.0,NaN,None,None,None,5433899.0,living,25844.0,8226026,06073012501,h564,SDGE,RI104,40.0,1895.486015,12525.523059,12525.523059,NaN,POINT (-117.09531 32.63634),NaN


In [154]:
# ensure no total_unit values are na
complete_zillow[complete_zillow['total_unit'].isna()]

# ensure no total_unit values are below 1
complete_zillow[complete_zillow['total_unit']<1]

,PARNO,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,total_unit,area_m2,volume_m3,total_volume_m3,missing_total_volume_m3,geometry


In [58]:
print(f"Number of Multi-family homes that have a unit of zero:", len(complete_zillow[complete_zillow['total_unit']<1]))

Number of Multi-family homes that have a unit of zero: 0


## Final results

## Renaming and saving

In [59]:
# save the complete Zillow points
complete_zillow_sd = complete_zillow.copy()

In [ ]:
# takes a really long time :,(
complete_zillow_sd.to_file("data/complete_zillow_sd.geojson", driver='GeoJSON')

KeyboardInterrupt: 